In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import timm
import math

from torch.utils.tensorboard import SummaryWriter

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
mean    = [0.485, 0.456, 0.406]
std     = [0.229, 0.224, 0.225]

aug_l   = 2
mag     = 15
alpha   = 0.2

batch_size = 128
num_workers = 2

# Set up data transformations
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    # transforms.RandAugment(magnitude=mag, num_ops=aug_l),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [4]:
import copy

# Load CIFAR-10 dataset
dataset = datasets.CIFAR100(root='C:\Datasets', train=True, download=True, transform=train_transform)

train_size = int(0.98 * len(dataset))
val_size = len(dataset) - train_size

trainset, valset= torch.utils.data.random_split(dataset, [train_size, val_size])
valset = copy.deepcopy(valset)

valset.dataset.transform   = val_transform

testset = datasets.CIFAR100(root='C:\Datasets', train=False, download=True, transform=val_transform)


train_loader = DataLoader(
    trainset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=num_workers, 
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    valset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers, 
    pin_memory=True,
    persistent_workers=True,
)

# test_loader = DataLoader(
#     testset, 
#     batch_size=batch_size, 
#     shuffle=False, 
#     num_workers=num_workers, 
#     pin_memory=True,
#     persistent_workers=True
# )

Files already downloaded and verified
Files already downloaded and verified


In [5]:
# timm.list_models()

In [13]:
class SE(nn.Module):
    def __init__(self, in_channels, out_channels = 1000, reduction_ratio=4):
        super(SE, self).__init__()
        if not out_channels:
            out_channels = in_channels
        self.fc1 = nn.Linear(in_channels, int(in_channels // reduction_ratio))
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(int(in_channels // reduction_ratio), out_channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

In [14]:
def create_model(model_name, pretrained, use_SE, to_freeze=False, num_classes=100, ratio = 4):
    # Define the model
    
    # model = timm.create_model("resnet50", pretrained=pretrained, num_classes=10)
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)

    if use_SE:
        if model_name == 'convit_small':
            new_head = nn.Sequential(SE(432),
                                     model.head)
        elif model_name in ["vit_small_patch16_224_in21k", 'vit_small_patch16_224']:
            new_head = nn.Sequential(SE(384, reduction_ratio=ratio),
                                     nn.Linear(1000, 100))
            # new_head = nn.Sequential(SE(384),
            #                          model.head)
        elif model_name == "vit_base_patch16_224_in21k":
            new_head = nn.Sequential(SE(768),
                                     model.head)
            # new_head = nn.Sequential(SE(768),
            #                          model.head)
        elif model_name == "coatnet_0_rw_224":
            model.head.flatten = nn.Sequential(
                                    model.head.fc,
                                    model.head.flatten,
            )
            model.head.fc = SE(768)
            new_head = model.head
        model.head = new_head
    
    if to_freeze:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.head.parameters():
            param.requires_grad = True
    model = model.to(device)

    return model

In [15]:
# model_name = "vit_small_patch16_224_in21k"
model_name = 'vit_small_patch16_224'
# model_name = "vit_base_patch16_224_in21k"
# model_name = "convit_small"
# model_name = "coatnet_0_rw_224"

notes = ''
freeze = False
if freeze:
    notes += '_Freeze'

tb_write = True

ratios = [0.5]
use_SEs = [True]
NUM_ACCUMULATION_STEPS = 1
lrs = [0.03]
lr = 0.03

In [16]:
for use_SE in use_SEs:
    for ratio in ratios:
        model = create_model(model_name, True, use_SE, to_freeze=freeze, ratio=ratio)

        steps = 1500
        warmup_steps = steps * 0.1

        num_epochs = math.ceil(steps / len(train_loader)) * NUM_ACCUMULATION_STEPS
        
        loss_fn = nn.CrossEntropyLoss()
        # optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-5)
        # optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=0.1)
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

        # Define loss function and optimizer
        warmup_scheduler = LinearLR(
            optimizer,
            start_factor=1/warmup_steps,
            end_factor=1, 
            total_iters=warmup_steps
        )
        cosine_scheduler = CosineAnnealingLR(
            optimizer,
            T_max=steps-warmup_steps,
            eta_min=0
        )

        scheduler = SequentialLR(
            optimizer, 
            [warmup_scheduler, cosine_scheduler],
            milestones=[warmup_steps]
        )

        if tb_write:
            writer = SummaryWriter(comment=f'cifar100_{model_name}_bs{batch_size}_accum{NUM_ACCUMULATION_STEPS}_steps{steps}_warm{warmup_steps}_lr{lr}_useSE{use_SE}{notes}_ratio{ratio}')
        # writer.add_hparams(hparam_dict=hparams, metric_dict={'acc': 0})

        #@title training steps
        best_model = None
        best_val_loss = float('inf')

        use_amp = True
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        pbar = tqdm(total=steps, smoothing = 0)
        step = 1

        for epoch in range(num_epochs):
            for idx, (inputs, labels) in enumerate(train_loader):
                model.train()

                inputs, labels = inputs.to(device), labels.to(device)

                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                    outputs = model(inputs)
                    loss    = loss_fn(outputs, labels) / NUM_ACCUMULATION_STEPS
                
                scaler.scale(loss).backward()

                if ((idx + 1) % NUM_ACCUMULATION_STEPS == 0) or (idx + 1 == len(train_loader)):
                    scaler.unscale_(optimizer)

                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    pbar.set_description(f'TLoss: {loss}')
                    
                    scaler.step(optimizer)
                    scaler.update()

                    optimizer.zero_grad()
                    # Update the learning rate scheduler
                    scheduler.step()
                
                if tb_write:
                    writer.add_scalar("Loss/train", loss.item(), step)
                    writer.add_scalar("lr", scheduler.get_last_lr()[0], step)

                if step % 100 == 0 and ((idx + 1) % NUM_ACCUMULATION_STEPS == 0) or (idx + 1 == len(train_loader)):
                    # Evaluation phase
                    model.eval()
                    val_loss = 0.0
                    val_correct = 0
                    
                    with torch.no_grad():
                        for inputs, labels in val_loader:
                            inputs, labels = inputs.to(device), labels.to(device)

                            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                                outputs = model(inputs)
                                loss = loss_fn(outputs, labels)

                            val_loss += loss.item() * inputs.size(0)
                            val_correct += torch.sum(torch.argmax(outputs, dim=1) == labels)

                        val_loss /= len(val_loader.dataset)
                        val_accuracy = val_correct.float() / len(val_loader.dataset)
                    
                        if tb_write:
                            writer.add_scalar("Loss/test", val_loss, step)
                            writer.add_scalar("acc", val_accuracy, step)
                if ((idx + 1) % NUM_ACCUMULATION_STEPS == 0) or (idx + 1 == len(train_loader)):
                    pbar.update(1)
                    if(step == steps):
                        break
                    step += 1
                # if val_loss < best_val_loss:
                #     best_val_loss = val_loss
                #     best_model = model.state_dict()
                #     print('!!! New best model found !!!')
                #     # torch.save(best_model, 'best_model.pth')

        # torch.save(model, f'E:/Models/{model_name}_bs{batch_size}_lr{lr}_useSE{use_SE}.pth')

  0%|          | 0/1500 [00:00<?, ?it/s]

c:\Users\Main\anaconda3\lib\site-packages\torch\optim\lr_scheduler.py:163: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


In [16]:
torch.save(model, f'E:/Models/cifar100/{model_name}_bs{batch_size}_accum{NUM_ACCUMULATION_STEPS}_steps{steps}_warm{warmup_steps}_lr{lr}_useSE{use_SE}{notes}_ratio{ratio}.pth')

In [ ]:
# torch.save(model, 'E:/Models/sevit_small.pth')